Gemini API

In [4]:
%pip install google-genai

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import requests
import json
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "gemma2:2b"

emoji_list = "0: 😜, 1: 📸, 2: 😍, 3: 😂, 4: 😉, 5: 🎄, 6: 📷, 7: 🔥, 8: 😘, 9: ❤, 10: 😁, 11: 🇺🇸, 12: ☀, 13: ✨, 14: 💙, 15: 💕, 16: 😎, 17: 😊, 18: 💜, 19: 💯"

def process_local_chunk(df_chunk, worker_name, pos_idx, part_filename):
    # إذا كان هناك حفظ احتياطي قديم لهذا الجزء، حمله وأكمل عليه
    if os.path.exists(part_filename):
        df_part = pd.read_csv(part_filename)
        if 'LLM_Cleaned_Label' not in df_part.columns:
            df_part['LLM_Cleaned_Label'] = None
    else:
        df_part = df_chunk.copy()
        df_part['LLM_Cleaned_Label'] = None

    records = df_part.to_dict('records')
    
    pbar = tqdm(enumerate(records), total=len(records), desc=worker_name, position=pos_idx, leave=True)
    for idx, row in pbar:
        # تخطي الأسطر المصنفة مسبقاً في هذا الجزء
        if pd.notna(df_part.at[idx, 'LLM_Cleaned_Label']) and str(df_part.at[idx, 'LLM_Cleaned_Label']).strip() != "":
            continue
            
        safe_text = str(row['TEXT']).replace('\n', ' ').replace('\r', ' ').strip()
        
        prompt = f"""You are an expert NLP data annotator. Analyze this tweet and select the single most logical emoji index from this list:
        {emoji_list}
        
        Tweet: {safe_text}
        
        Return ONLY the integer index number (0-19). No words, no markdown.
        Index:"""

        payload = {
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.0}
        }
        
        predicted_index = 3  # القيمة الافتراضية عند حدوث خطأ نادر جداً
        try:
            response = requests.post(OLLAMA_URL, json=payload, timeout=10)
            output_text = response.json().get("response", "").strip()
            
            # استخراج الرقم الصافي
            clean_word = ''.join(filter(str.isdigit, output_text))
            if clean_word and int(clean_word) in range(20):
                predicted_index = int(clean_word)
            else:
                for word in output_text.split():
                    digit_check = ''.join(filter(str.isdigit, word))
                    if digit_check and int(digit_check) in range(20):
                        predicted_index = int(digit_check)
                        break
        except:
            pass
            
        df_part.at[idx, 'LLM_Cleaned_Label'] = predicted_index
        
        # حفظ احتياطي تلقائي كل 20 سطر لتأمين الشغل داخل هذا الجزء
        if idx % 20 == 0:
            df_part.to_csv(part_filename, index=False)
            
    df_part.to_csv(part_filename, index=False)
    return df_part

if __name__ == "__main__":
    input_file = 'data/Test.csv'
    output_file = 'data/Test_Local_LLM_Cleaned_15000.csv'
    
    # إنشاء مجلد مؤقت لحفظ الأجزاء فيه إن لم يكن موجوداً
    os.makedirs('data/metadata_chunks', exist_ok=True)
    
    print("Loading 15,000 rows for secure parallel local processing...")
    df_total = pd.read_csv(input_file).dropna(subset=['TEXT']).reset_index(drop=True)
    df_total = df_total.head(15000)
    
    total_rows = len(df_total)
    chunk_size = total_rows // 4
    
    # تقسيم الـ 15 ألف سطر على 4 أجزاء بالتساوي
    df_p1 = df_total.iloc[0:chunk_size].reset_index(drop=True)
    df_p2 = df_total.iloc[chunk_size:chunk_size*2].reset_index(drop=True)
    df_p3 = df_total.iloc[chunk_size*2:chunk_size*3].reset_index(drop=True)
    df_p4 = df_total.iloc[chunk_size*3:].reset_index(drop=True)
    
    part_files = [
        'data/metadata_chunks/part_1.csv',
        'data/metadata_chunks/part_2.csv',
        'data/metadata_chunks/part_3.csv',
        'data/metadata_chunks/part_4.csv'
    ]
    
    print("🚀 Launching 4 Local Workers concurrently on your GPU... Full Throttle!")
    
    with ThreadPoolExecutor(max_workers=4) as executor:
        f1 = executor.submit(process_local_chunk, df_p1, "Worker Alpha", 0, part_files[0])
        f2 = executor.submit(process_local_chunk, df_p2, "Worker Beta", 1, part_files[1])
        f3 = executor.submit(process_local_chunk, df_p3, "Worker Gamma", 2, part_files[2])
        f4 = executor.submit(process_local_chunk, df_p4, "Worker Delta", 3, part_files[3])
        
        res1 = f1.result()
        res2 = f2.result()
        res3 = f3.result()
        res4 = f4.result()
        
    # دمج الأجزاء الـ 4 بعد الاكتمال التام
    final_df = pd.concat([res1, res2, res3, res4], ignore_index=True)
    final_df.to_csv(output_file, index=False)
    
    print("\n" + "="*60)
    print(f"🎉 SUCCESS! All 15,000 rows processed locally with high quality.")
    print(f"Final file securely saved to: '{output_file}'")
    print("="*60)
    
    print("\n📊 Check your Local Labels Distribution:")
    print(final_df['LLM_Cleaned_Label'].value_counts())

Loading 15,000 rows for secure parallel local processing...
🚀 Launching 4 Local Workers concurrently on your GPU... Full Throttle!


Worker Alpha:   0%|          | 0/3750 [00:00<?, ?it/s]


Worker Alpha:   0%|          | 1/3750 [00:08<8:35:05,  8.24s/it]


Worker Alpha:   0%|          | 2/3750 [00:10<4:56:49,  4.75s/it]


Worker Alpha:   0%|          | 3/3750 [00:12<3:50:10,  3.69s/it]


Worker Alpha:   0%|          | 4/3750 [00:15<3:18:07,  3.17s/it]


Worker Alpha:   0%|          | 5/3750 [00:17<3:01:03,  2.90s/it]


Worker Alpha:   0%|          | 6/3750 [00:20<2:50:24,  2.73s/it]


Worker Alpha:   0%|          | 7/3750 [00:22<2:43:26,  2.62s/it]


Worker Alpha:   0%|          | 8/3750 [00:24<2:39:12,  2.55s/it]


Worker Alpha:   0%|          | 9/3750 [00:27<2:36:02,  2.50s/it]


Worker Alpha:   0%|          | 10/3750 [00:29<2:32:54,  2.45s/it]


Worker Alpha:   0%|          | 11/3750 [00:32<2:31:40,  2.43s/it]


Worker Alpha:   0%|          | 12/3750 [00:34<2:31:27,  2.43s/it]


Worker Alpha:   0%|          | 13/3750 [00:36<2:31:31,  2.43s/it]


Worker Alpha:   0%|          | 14/3750 [00:39<2:30:55,  2.42s/it]





🎉 SUCCESS! All 15,000 rows processed locally with high quality.
Final file securely saved to: 'data/Test_Local_LLM_Cleaned_15000.csv'

📊 Check your Local Labels Distribution:
LLM_Cleaned_Label
10    6162
2     5375
12     764
15     619
1      406
0      303
3      296
6      240
13     144
8      121
18     115
7      101
16      78
14      78
11      65
17      53
4       46
5       27
9        6
19       1
Name: count, dtype: int64


In [8]:
import pandas as pd

# تحميل ملف الـ 15 ألف المطحون وجاهز
df_old = pd.read_csv('data/Test_Local_LLM_Cleaned_15000.csv')

# قائمة الـ 10 كلاسات الذهبية المعتمدة
allowed_classes = [10, 2, 12, 15, 1, 0, 3, 13, 8, 7]

# فلترة الداتا بلحظة
df_10k = df_old[df_old['LLM_Cleaned_Label'].isin(allowed_classes)].reset_index(drop=True)

# حفظ الملف الجديد النقي تماماً بـ 10 كلاسات
df_10k.to_csv('data/Test_Local_Cleaned_10Classes.csv', index=False)

print(f"New dataset shape: {df_10k.shape} rows ready for action!")

New dataset shape: (14291, 4) rows ready for action!


In [19]:
import os
import re
import pickle
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    DistilBertTokenizerFast, 
    DistilBertForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def clean_tweet_text(text):
    text = str(text).lower()
    text = re.sub(r"http\s+", "", text)              
    text = re.sub(r"www\.\S+", "", text)             
    text = re.sub(r"@\w+", "", text)                 
    text = re.sub(r"&amp;|&lt;|&gt;", "", text)       
    text = re.sub(r"\s+", " ", text).strip()         
    return text

print("Loading dataset for Top 9 perfect classes...")
df = pd.read_csv('data/Test_Local_LLM_Cleaned_15000.csv')
df = df.dropna(subset=['TEXT', 'LLM_Cleaned_Label'])
df['LLM_Cleaned_Label'] = df['LLM_Cleaned_Label'].astype(int)

allowed_classes = [10, 2, 12, 15, 1, 0, 3, 13, 7]
df_9k = df[df['LLM_Cleaned_Label'].isin(allowed_classes)].reset_index(drop=True)

df_9k['CLEANED_TEXT'] = df_9k['TEXT'].apply(clean_tweet_text)

label_to_mapped = {orig: mapped for mapped, orig in enumerate(allowed_classes)}
mapped_to_label = {mapped: orig for mapped, orig in enumerate(allowed_classes)}
df_9k['Mapped_Label'] = df_9k['Mapped_Label'] = df_9k['LLM_Cleaned_Label'].map(label_to_mapped)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_9k['CLEANED_TEXT'].tolist(), 
    df_9k['Mapped_Label'].tolist(), 
    test_size=0.15, 
    random_state=42
)

print("Computing balanced class weights for 9 classes...")
all_mapped_classes = np.arange(9)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=all_mapped_classes,
    y=train_labels
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

class EmojiTrainDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = EmojiTrainDataset(train_encodings, train_labels)
val_dataset = EmojiTrainDataset(val_encodings, val_labels)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'f1_macro': f1}

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=9)

class BalancedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# تعديل مسار المخرجات لضمان النقاء التجريبي كامل
training_args = TrainingArguments(
    output_dir='./results_perfect_balanced_9classes',
    num_train_epochs=5,               
    per_device_train_batch_size=16,   
    per_device_eval_batch_size=32,
    learning_rate=3e-5,               
    lr_scheduler_type="cosine",       
    warmup_steps=200,                 
    weight_decay=0.01,
    logging_dir='./logs_perfect_balanced_9classes',
    logging_steps=50,
    eval_strategy="epoch",            
    save_strategy="epoch",            
    load_best_model_at_end=True,      
    metric_for_best_model="accuracy", 
    fp16=True,                        
)

trainer = BalancedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

print("\n🔥 Training the Ultimate Balanced 9-Class Model...")
trainer.train()

# حفظ في مجلد منفصل ونظيف تماماً باسم 9classes
new_model_path = "./best_emoji_distilbert_9classes"
model.save_pretrained(new_model_path)
tokenizer.save_pretrained(new_model_path)

with open(f"{new_model_path}/mapped_to_label.pkl", "wb") as f:
    pickle.dump(mapped_to_label, f)

print(f"\n🎉 SUCCESS! Fresh 9-Class Model saved cleanly to '{new_model_path}'")

Using device: cuda
Loading dataset for Top 9 perfect classes...
Computing balanced class weights for 9 classes...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3967.63it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



🔥 Training the Ultimate Balanced 9-Class Model...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.495600,1.697119,0.562559,0.356548
2,1.153264,1.607697,0.569144,0.397627
3,0.903121,1.834155,0.591251,0.394980
4,0.536455,2.180640,0.610536,0.399217
5,0.427232,2.252969,0.616181,0.401256


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.47it/s]


🎉 SUCCESS! Fresh 9-Class Model saved cleanly to './best_emoji_distilbert_9classes'


In [4]:
import os
import re
import pickle
import torch
import pandas as pd
from tqdm import tqdm
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from torch.utils.data import DataLoader, Dataset

# 1. تفعيل كرت الشاشة للسرعة القصوى
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# دالة التنظيف المعتمدة للمشروع
def clean_tweet_text(text):
    text = str(text).lower()
    text = re.sub(r"http\s+", "", text)              
    text = re.sub(r"www\.\S+", "", text)             
    text = re.sub(r"@\w+", "", text)                 
    text = re.sub(r"&amp;|&lt;|&gt;", "", text)       
    text = re.sub(r"\s+", " ", text).strip()         
    return text

# 2. 🚀 السحر هنا: تحميل ملفات الـ Train والـ Test ودمجهم للوصول للـ 70 ألف سطر كاملين
print("Loading Train.csv and Test.csv...")
df_train = pd.read_csv('data/Train.csv')
df_test = pd.read_csv('data/Test.csv')

print(f"-> Train rows: {len(df_train)}, Test rows: {len(df_test)}")

# دمج الملفين فوق بعض في جدول واحد عملاق
df_full = pd.concat([df_train, df_test], ignore_index=True)
print(f"💪 Total Combined Dataset Size: {len(df_full)} rows!")

# معالجة الفراغات لمنع الـ Tokenizer من الانهيار
df_full['TEXT'] = df_full['TEXT'].fillna("").astype(str)

print("Applying Regex Text Cleaning to all rows...")
cleaned_texts = df_full['TEXT'].apply(clean_tweet_text).tolist()

# 3. تحميل الموديل الخبير صاحب دقة 79.66% والـ Tokenizer
model_path = "./final_production_emoji_model"
print(f"Loading your high-accuracy model from: '{model_path}'...")
tokenizer = DistilBertTokenizerFast.from_pretrained(model_path)
model = DistilBertForSequenceClassification.from_pretrained(model_path).to(device)
model.eval()  

# القاموس المعتمد والآمن للـ 9 كلاسات الصافية
allowed_classes = [10, 2, 12, 15, 1, 0, 3, 13, 7]
mapped_to_label = {mapped: orig for mapped, orig in enumerate(allowed_classes)}

# 4. الـ Tokenization والـ Dataset للـ 70 ألف سطر
print("Tokenizing the entire 70K dataset... Please wait a moment...")
encodings = tokenizer(cleaned_texts, truncation=True, padding=True, max_length=128)

class InferenceDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
    def __len__(self):
        return len(self.encodings['input_ids'])

inference_dataset = InferenceDataset(encodings)
# batch_size=256 طيارة وآمن جداً على كرت الـ 4GB VRAM أثناء الـ Inference
data_loader = DataLoader(inference_dataset, batch_size=256, shuffle=False)

# 5. تشغيل الـ Pipeline وعصر كرت الشاشة
print("🚀 Launching Ultimate Turbo Inference Pipeline over all rows...")
mapped_preds = []

with torch.no_grad():
    for batch in tqdm(data_loader, desc="Predicting Emojis"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        with torch.amp.autocast('cuda'):  
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            
        preds = outputs.logits.argmax(dim=-1).cpu().numpy()
        mapped_preds.extend(preds)

print("Decoding predictions back to original emoji labels...")
final_labels = [mapped_to_label[p] for p in mapped_preds]

# 6. دمج النتائج الجديدة وحفظ الملف المليونير النهائي للمشروع
df_full['LLM_Cleaned_Label'] = final_labels
output_final_file = 'data/Final_70K_Perfect_Cleaned.csv'
df_full.to_csv(output_final_file, index=False)

print("\n" + "="*60)
print("🎉 GOLDEN TRIPLE SUCCESS! The entire 70K dataset is now 100% classified!")
print(f"Saved securely to: '{output_final_file}'")
print("="*60)

print("\n📊 Final Predicted Emojis Distribution across your TRUE 70K rows:")
print(df_full['LLM_Cleaned_Label'].value_counts())
print(f"\nTotal rows verified in file: {len(df_full)}")

Using device: cuda
Loading Train.csv and Test.csv...
-> Train rows: 70000, Test rows: 25958
💪 Total Combined Dataset Size: 95958 rows!
Applying Regex Text Cleaning to all rows...
Loading your high-accuracy model from: './final_production_emoji_model'...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 4691.56it/s]


Tokenizing the entire 70K dataset... Please wait a moment...
🚀 Launching Ultimate Turbo Inference Pipeline over all rows...


Predicting Emojis: 100%|██████████| 375/375 [07:53<00:00,  1.26s/it]


Decoding predictions back to original emoji labels...

🎉 GOLDEN TRIPLE SUCCESS! The entire 70K dataset is now 100% classified!
Saved securely to: 'data/Final_70K_Perfect_Cleaned.csv'

📊 Final Predicted Emojis Distribution across your TRUE 70K rows:
LLM_Cleaned_Label
2     35512
10    34521
12     8285
15     6484
1      4288
3      2604
0      2233
13     1173
7       858
Name: count, dtype: int64

Total rows verified in file: 95958


In [21]:
import os
import re
import pickle
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    DistilBertTokenizerFast, 
    DistilBertForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback
)

# 1. تفقد كرت الشاشة لتوجيه الطاقة القصوى له
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# دالة التنظيف المعتمدة الموحدة للمشروع
def clean_tweet_text(text):
    text = str(text).lower()
    text = re.sub(r"http\s+", "", text)              
    text = re.sub(r"www\.\S+", "", text)             
    text = re.sub(r"@\w+", "", text)                 
    text = re.sub(r"&amp;|&lt;|&gt;", "", text)       
    text = re.sub(r"\s+", " ", text).strip()         
    return text

# 2. تحميل الداتا الجديدة الموزونة بالكامل (الـ 26 ألف سطر المقفلة)
print("Loading the newly labeled production dataset...")
df = pd.read_csv('data/Final_70K_Perfect_Cleaned.csv')
df = df.dropna(subset=['TEXT', 'LLM_Cleaned_Label'])
df['LLM_Cleaned_Label'] = df['LLM_Cleaned_Label'].astype(int)

print("Applying Text Cleaning Pipeline to the dataset...")
df['CLEANED_TEXT'] = df['TEXT'].apply(clean_tweet_text)

# الـ Mapping الصارم للـ 9 كلاسات المعتمدة
allowed_classes = [10, 2, 12, 15, 1, 0, 3, 13, 7]
label_to_mapped = {orig: mapped for mapped, orig in enumerate(allowed_classes)}
mapped_to_label = {mapped: orig for mapped, orig in enumerate(allowed_classes)}
df['Mapped_Label'] = df['LLM_Cleaned_Label'].map(label_to_mapped)

# 3. تقسيم الداتا لتدريب (85%) وتقييم (15%)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['CLEANED_TEXT'].tolist(), 
    df['Mapped_Label'].tolist(), 
    test_size=0.15, 
    random_state=42
)

# 4. حساب الأوزان بناءً على التوزيع الموزون الجديد لضمان ثبات الأداء
print("Computing balanced class weights for final tuning...")
all_mapped_classes = np.arange(9)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=all_mapped_classes,
    y=train_labels
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

# 5. تجهيز الـ Tokenizer والـ Dataset لهيكلية PyTorch
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

class EmojiTrainDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = EmojiTrainDataset(train_encodings, train_labels)
val_dataset = EmojiTrainDataset(val_encodings, val_labels)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'f1_macro': f1}

# 6. تحميل نسخة خام ونظيفة من الموديل لبنائها كنسخة نهائية للإنتاج
print("Initializing Final Production DistilBERT Model (9 Targeted Classes)...")
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=9)

class BalancedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# 7. إعدادات التدريب النهائي المصقول والآمن
training_args = TrainingArguments(
    output_dir='./results_final_production',
    num_train_epochs=3,               # 3 جولات مثالية جداً ومستقرة لهذا الحجم من البيانات
    per_device_train_batch_size=16,   
    per_device_eval_batch_size=32,
    learning_rate=3e-5,               # معدل هادئ وموزون لمنع الـ Overfitting السريع
    lr_scheduler_type="cosine",       
    warmup_steps=300,                 
    weight_decay=0.01,
    logging_dir='./logs_final_production',
    logging_steps=50,
    eval_strategy="epoch",            
    save_strategy="epoch",            
    load_best_model_at_end=True,      
    metric_for_best_model="accuracy", # التركيز على دقة الاستقرار الكلية
    fp16=True,                        
)

trainer = BalancedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)] # حماية تلقائية فورية
)

print("\n🔥 Launching Final Production Fine-Tuning Loop on GPU... Here we go!")
trainer.train()

# 8. الحفظ النهائي للموديل المقفل والقاموس الخاص به في المجلد الجديد
final_model_path = "./final_production_emoji_model"
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

with open(f"{final_model_path}/mapped_to_label.pkl", "wb") as f:
    pickle.dump(mapped_to_label, f)

print(f"\n🎉 ALL DONE! Your final production model has been trained and saved securely to: '{final_model_path}'")

Using device: cuda
Loading the newly labeled production dataset...
Applying Text Cleaning Pipeline to the dataset...
Computing balanced class weights for final tuning...
Initializing Final Production DistilBERT Model (9 Targeted Classes)...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5662.62it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



🔥 Launching Final Production Fine-Tuning Loop on GPU... Here we go!


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.062249,0.934387,0.669235,0.564100
2,0.545352,0.871316,0.765537,0.667079
3,0.287999,0.965125,0.796610,0.687916


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.76it/s]


🎉 ALL DONE! Your final production model has been trained and saved securely to: './final_production_emoji_model'


In [4]:
# رن هذه الخلية داخل النوت بوك فوراً لتثبيت المكتبات بالبيئة الصحيحة
# تشغيل هذا الأمر لتحديث بايتورش لنسخة آمنة فوق 2.6 ومتوافقة مع كرت الشاشة
%pip install --upgrade torch torchvision torchaudio

  Using cached torch-2.12.0-cp311-cp311-win_amd64.whl.metadata (31 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
Using cached torch-2.12.0-cp311-cp311-win_amd64.whl (123.0 MB)
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.7 MB 3.4 MB/s eta 0:00:01
   ------------- -------------------------- 1.3/3.7 MB 3.9 MB/s eta 0:00:01
   ------------------------- -------------- 2.4/3.7 MB 4.5 MB/s eta 0:00:01
   ------------------------------------ --- 3.4/3.7 MB 4.4 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 4.2 MB/s  0:00:01
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)

  Attempting uninstall: torchaudio

    Found existing installation: torchaudio 2.5.1+cu121

    Uninstalling torchaudio-2.5.1+cu121:

      Successfully uninstalled torchaudio-2.5.1+cu121

   ---------------------------------------- 0/

  You can safely remove it manually.
  You can safely remove it manually.


In [3]:
# 1. تنزيل أداة uv الصاروخية بلمحة بصر
%pip install uv

# 2. إطلاق التحميل المتوازي لـ PyTorch CUDA بكامل سرعة الإنترنت عندك
!uv pip install --system torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

   ---------------------------------------- 0.0/25.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/25.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/25.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/25.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/25.2 MB ? eta -:--:--
    --------------------------------------- 0.5/25.2 MB 699.0 kB/s eta 0:00:36
    --------------------------------------- 0.5/25.2 MB 699.0 kB/s eta 0:00:36
    --------------------------------------- 0.5/25.2 MB 699.0 kB/s eta 0:00:36
    --------------------------------------- 0.5/25.2 MB 699.0 kB/s eta 0:00:36
   - -------------------------------------- 0.8/25.2 MB 493.7 kB/s eta 0:00:50
   - -------------------------------------- 0.8/25.2 MB 493.7 kB/s eta 0:00:50
   - -------------------------------------- 0.8/25.2 MB 493.7 kB/s eta 0:00:50
   - -------------------------------------- 0.8/25.2 MB 493.7 kB/s eta 0:00:50
   - -------------

Using Python 3.11.5 environment at: C:\Users\user\miniconda3
Resolved 13 packages in 13.18s
 Downloaded sympy
 Downloaded torchaudio
 Downloaded torchvision
 Downloaded torch
Prepared 4 packages in 13m 54s
Uninstalled 1 package in 2ms
Installed 4 packages in 30.20s
 - sympy==1.14.0
 + sympy==1.13.1
 + torch==2.5.1+cu121
 + torchaudio==2.5.1+cu121
 + torchvision==0.20.1+cu121


In [3]:
import os
import re
import pickle
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback
)

# 1. تفقد الـ CUDA لتوجيه طاقة الـ GPU كاملة
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# دالة التنظيف الموحدة لضمان نقاء النصوص
def clean_tweet_text(text):
    text = str(text).lower()
    text = re.sub(r"http\s+", "", text)              
    text = re.sub(r"www\.\S+", "", text)             
    text = re.sub(r"@\w+", "", text)                 
    text = re.sub(r"&amp;|&lt;|&gt;", "", text)       
    text = re.sub(r"\s+", " ", text).strip()         
    return text

# 2. تحميل الداتا الموزونة والمقفلة
print("Loading the production dataset...")
df = pd.read_csv('data/Final_70K_Perfect_Cleaned.csv')
df = df.dropna(subset=['TEXT', 'LLM_Cleaned_Label'])
df['LLM_Cleaned_Label'] = df['LLM_Cleaned_Label'].astype(int)

print("Cleaning texts for DeBERTa pipeline...")
df['CLEANED_TEXT'] = df['TEXT'].apply(clean_tweet_text)

# الـ Mapping المعتمد للـ 9 كلاسات
allowed_classes = [10, 2, 12, 15, 1, 0, 3, 13, 7]
label_to_mapped = {orig: mapped for mapped, orig in enumerate(allowed_classes)}
mapped_to_label = {mapped: orig for mapped, orig in enumerate(allowed_classes)}
df['Mapped_Label'] = df['LLM_Cleaned_Label'].map(label_to_mapped)

# 3. تقسيم الداتا (85% تدريب، 15% تقييم)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['CLEANED_TEXT'].tolist(), 
    df['Mapped_Label'].tolist(), 
    test_size=0.15, 
    random_state=42
)

# 4. حساب أوزان الكلاسات لفرملة الانحياز
print("Computing balanced class weights...")
all_mapped_classes = np.arange(9)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=all_mapped_classes,
    y=train_labels
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

# 5. تحميل الـ Tokenizer الخاص بـ DeBERTa-v3
model_name = "microsoft/deberta-v3-base"
print(f"Loading Tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False) 

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

class EmojiDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = EmojiDataset(train_encodings, train_labels)
val_dataset = EmojiDataset(val_encodings, val_labels)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'f1_macro': f1}

# 6. تحميل موديل DeBERTa-v3-base لـ 9 كلاسات وتجهيز الـ Trainer المحسن
print(f"Initializing {model_name} with 9 classification heads...")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=9, use_safetensors=True)

class BalancedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits").float() 
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# 7. إعدادات تدريب مستقرة 100% بالدقة الكاملة وحماية الـ VRAM
training_args = TrainingArguments(
    output_dir='./results_deberta_production',
    num_train_epochs=4,               
    per_device_train_batch_size=8,    
    gradient_accumulation_steps=2,    
    per_device_eval_batch_size=16,   
    learning_rate=2e-5,               
    lr_scheduler_type="cosine",       
    warmup_steps=200,                 
    weight_decay=0.01,
    logging_dir='./logs_deberta',
    logging_steps=50,
    eval_strategy="epoch",            
    save_strategy="epoch",            
    load_best_model_at_end=True,      
    metric_for_best_model="accuracy", 
    bf16=False,                       # 🔥 إيقاف الـ BF16 نهائياً لحل مشكلة الـ NaN وعودة الاستقرار
    fp16=False,                       # إيقاف الـ FP16
    gradient_checkpointing=True,      # 🔥 تفعيل تقنية حظر ضغط الـ VRAM لمنع الـ OOM في الـ Float32
)

trainer = BalancedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

print("\n🚀 Launching Stable DeBERTa-v3 Fine-Tuning Loop (Full Precision Float32)... Let's do it right!")
trainer.train()

# 8. الحفظ في مسار مستقل وفخم للسي دي
deberta_path = "./final_deberta_production_model"
model.save_pretrained(deberta_path)
tokenizer.save_pretrained(deberta_path)

with open(f"{deberta_path}/mapped_to_label.pkl", "wb") as f:
    pickle.dump(mapped_to_label, f)

print(f"\n🎉 GOLDEN SUCCESS! Your SOTA DeBERTa-v3 model is locked and saved to: '{deberta_path}'")

Using device: cuda
Loading the production dataset...
Cleaning texts for DeBERTa pipeline...
Computing balanced class weights...
Loading Tokenizer for microsoft/deberta-v3-base...
Initializing microsoft/deberta-v3-base with 9 classification heads...


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 4129.22it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    


🚀 Launching Stable DeBERTa-v3 Fine-Tuning Loop (Full Precision Float32)... Let's do it right!


Epoch,Training Loss,Validation Loss
